# Notebook 10b — 3B Ceiling Baseline · ARC-Challenge
## SLM-to-SLM Guided Reasoning — Compute Ceiling Comparison

**Purpose:** Run Qwen2.5-3B alone with 5 votes — no fine-tuning, no LoRA, no guide.
This is the **compute ceiling** (15B param-passes) vs our pipeline's 10.5B.

| Condition | Setup | Compute |
|-----------|-------|---------|
| Baseline | 1.5B × 5 | 7.5B |
| **Our Pipeline** | 3B guide (LoRA) + 1.5B × 5 | **10.5B** |
| **← This notebook** | 3B base × 5 | **15.0B** |

Same 900 ARC-Challenge questions · Same seed=42 · Same temp=0.4 · Same 3-angle evaluation

**Why this matters:** ARC-Challenge is a key out-of-domain science reasoning result.
Does the raw 3B model, with no fine-tuning but more compute (15B vs 10.5B), beat the guided pipeline?
If pipeline wins at 30% lower cost → structured verbal planning is the sufficient condition, not raw model size.
If ceiling wins → model capacity explains the gains, not fine-tuned planning.


In [ ]:
# CELL 1 -- Install (uncomment on first run)
# !pip install -q transformers==4.44.0
# !pip install -q accelerate==0.33.0
# !pip install -q datasets==2.20.0
# !pip install -q huggingface_hub
print("Done.")

In [1]:
# CELL 2 -- HuggingFace login
from huggingface_hub import login
login("")
print('HuggingFace login done')

HuggingFace login done


In [2]:
# CELL 3 -- Imports + GPU check
import os, json, re, time, random
import torch
import numpy as np
from collections import Counter, defaultdict
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.notebook import tqdm

OUTPUT_DIR = "/kaggle/working/arc_ceiling"
os.makedirs(OUTPUT_DIR, exist_ok=True)

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"PyTorch : {torch.__version__}")
    print(f"GPU     : {props.name}")
    print(f"VRAM    : {props.total_memory/1024**3:.1f} GB")
else:
    print("No GPU detected")
print(f"Output  : {OUTPUT_DIR}")

PyTorch : 2.9.0+cu126
GPU     : Tesla P100-PCIE-16GB
VRAM    : 15.9 GB
Output  : /kaggle/working/arc_ceiling


In [3]:
# CELL 4 -- Configuration
# ONE model only. No guide. No LoRA. No fine-tuning.
# Ceiling condition: 3B x 5 votes = 15B param-passes.
CONFIG = {
    "model_name"          : "Qwen/Qwen2.5-3B-Instruct",
    "model_params_B"      : 3.0,
    # Dataset -- IDENTICAL to Notebook 10 (same split, same seed, same N)
    "dataset_name"        : "allenai/ai2_arc",
    "dataset_config"      : "ARC-Challenge",
    "dataset_split"       : "test",
    "max_eval_samples"    : 900,             # match N=900 pipeline run
    "random_seed"         : 42,             # FIXED -- must match Notebook 10 exactly
    # Voting
    "n_votes"             : 5,
    "vote_temperature"    : 0.4,            # same as pipeline N=900 run
    "refiner_temperature" : 0.3,
    "max_new_tokens"      : 400,
    # Random chance for ARC-Challenge (4 options A-D)
    "random_chance"       : 25.0,
    # Known results from Notebook 10 N=900 pipeline run (for Angle 1 three-way table)
    "known_baseline_acc"  : 71.7,           # 1.5B x 5 @ 7.5B, N=900
    "known_pipeline_acc"  : 80.0,           # 3B+LoRA + 1.5B x 5 @ 10.5B, N=900
    "known_baseline_B"    : 7.5,
    "known_pipeline_B"    : 10.5,
    # Output paths
    "results_file"        : f"{OUTPUT_DIR}/results.jsonl",
    "report_file"         : f"{OUTPUT_DIR}/eval_report.json",
    "angle1_file"         : f"{OUTPUT_DIR}/angle1_compute_efficiency.json",
    "angle2_file"         : f"{OUTPUT_DIR}/angle2_vote_consistency.json",
    "angle3_file"         : f"{OUTPUT_DIR}/angle3_confidence_calibration.json",
    "checkpoint_file"     : f"{OUTPUT_DIR}/checkpoint.json",
    "save_every"          : 25,
}
print("Config ready:")
for k, v in CONFIG.items():
    print(f"  {k:<26}: {v}")

Config ready:
  model_name                : Qwen/Qwen2.5-3B-Instruct
  model_params_B            : 3.0
  dataset_name              : allenai/ai2_arc
  dataset_config            : ARC-Challenge
  dataset_split             : test
  max_eval_samples          : 900
  random_seed               : 42
  n_votes                   : 5
  vote_temperature          : 0.4
  refiner_temperature       : 0.3
  max_new_tokens            : 400
  random_chance             : 25.0
  known_baseline_acc        : 71.7
  known_pipeline_acc        : 80.0
  known_baseline_B          : 7.5
  known_pipeline_B          : 10.5
  results_file              : /kaggle/working/arc_ceiling/results.jsonl
  report_file               : /kaggle/working/arc_ceiling/eval_report.json
  angle1_file               : /kaggle/working/arc_ceiling/angle1_compute_efficiency.json
  angle2_file               : /kaggle/working/arc_ceiling/angle2_vote_consistency.json
  angle3_file               : /kaggle/working/arc_ceiling/angle3_confidenc

In [4]:
# CELL 5 -- Load ARC-Challenge dataset
# Identical normalisation to Notebook 10 -- same seed, same sampling, same formatting.
# Fields: question, choices (label + text), answerKey
# We format as: "<question>\n\nOptions:\nA) ...\nB) ...\nC) ...\nD) ..."

NUM_TO_LETTER = {"1": "A", "2": "B", "3": "C", "4": "D", "5": "E"}
VALID_LETTERS = set("ABCD")   # ARC-Challenge is 4-option; occasionally A-E

def normalise_label(label):
    s = str(label).strip().upper()
    return NUM_TO_LETTER.get(s, s)

def normalise_arc(item):
    labels      = [normalise_label(l) for l in item["choices"]["label"]]
    texts       = item["choices"]["text"]
    options_str = "\n".join(f"{l}) {t}" for l, t in zip(labels, texts))
    q   = item["question"].strip() + "\n\nOptions:\n" + options_str
    ans = normalise_label(item["answerKey"])
    return {"question": q, "answer": ans, "raw_choices": list(zip(labels, texts))}

print("Loading ARC-Challenge from HuggingFace...")
raw_ds   = load_dataset(CONFIG["dataset_name"], CONFIG["dataset_config"])
all_data = [normalise_arc(x) for x in raw_ds[CONFIG["dataset_split"]]]

# Collect all valid letters found in actual data (usually A-D, rarely A-E)
all_labels = set()
for item in all_data:
    for lbl, _ in item["raw_choices"]:
        all_labels.add(lbl)
VALID_LETTERS = all_labels

print(f"Splits   : {list(raw_ds.keys())}")
print(f"Test size: {len(raw_ds[CONFIG['dataset_split']])}")
print(f"Formatted: {len(all_data)} questions")
print(f"Valid answer letters: {sorted(VALID_LETTERS)}")

# CRITICAL: same seed as Notebook 10
random.seed(CONFIG["random_seed"])
np.random.seed(CONFIG["random_seed"])
n = min(CONFIG["max_eval_samples"], len(all_data))
test_data = random.sample(all_data, n)

print(f"Sampled {len(test_data)} questions (seed={CONFIG['random_seed']})")
print(f"\nSample question:\n{test_data[0]['question']}")
print(f"Answer  : {test_data[0]['answer']}")

Loading ARC-Challenge from HuggingFace...


README.md: 0.00B [00:00, ?B/s]

ARC-Challenge/train-00000-of-00001.parqu(…):   0%|          | 0.00/190k [00:00<?, ?B/s]

ARC-Challenge/test-00000-of-00001.parque(…):   0%|          | 0.00/204k [00:00<?, ?B/s]

ARC-Challenge/validation-00000-of-00001.(…):   0%|          | 0.00/55.7k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1119 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1172 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/299 [00:00<?, ? examples/s]

Splits   : ['train', 'test', 'validation']
Test size: 1172
Formatted: 1172 questions
Valid answer letters: ['A', 'B', 'C', 'D', 'E']
Sampled 900 questions (seed=42)

Sample question:
When cold temperatures are produced in a chemical reaction, the reaction is known as

Options:
A) exothermic.
B) endothermic.
C) suspension.
D) vaporization.
Answer  : B


In [5]:
# CELL 6 -- Answer extraction for MCQ (A-D)
# Identical to Notebook 10 -- same priority order, same regex patterns.

def extract_gt_answer(answer_str):
    s = normalise_label(str(answer_str).strip())
    return s if s in VALID_LETTERS else ""

def extract_pred_answer(text):
    text = text.strip()
    # 1. Conclusive phrases: "the answer is X"
    m = re.search(
        r"(?:the answer is|answer is|answer:|the correct answer is|correct answer is)"
        r"[\s:]*([A-D])\b", text, re.IGNORECASE)
    if m and m.group(1).upper() in VALID_LETTERS: return m.group(1).upper()
    # 2. "option/choice X is correct"
    m = re.search(
        r"(?:option|choice)\s+([A-D])\s+(?:is correct|is the answer|matches|is right)",
        text, re.IGNORECASE)
    if m and m.group(1).upper() in VALID_LETTERS: return m.group(1).upper()
    # 3. #### A
    m = re.search(r"####\s*([A-D])\b", text, re.IGNORECASE)
    if m and m.group(1).upper() in VALID_LETTERS: return m.group(1).upper()
    # 4. (A) at end
    m = re.search(r"\(([A-D])\)\s*$", text, re.IGNORECASE)
    if m and m.group(1).upper() in VALID_LETTERS: return m.group(1).upper()
    # 5. **A**
    m = re.search(r"\*\*([A-D])\)?\*\*", text, re.IGNORECASE)
    if m and m.group(1).upper() in VALID_LETTERS: return m.group(1).upper()
    # 6. Standalone letter on its own line (last occurrence)
    matches = re.findall(r"^\s*([A-D])\s*$", text, re.MULTILINE | re.IGNORECASE)
    if matches: return matches[-1].upper()
    # 7. Last standalone letter anywhere
    matches = re.findall(r"\b([A-D])\b", text, re.IGNORECASE)
    if matches: return matches[-1].upper()
    return ""

# Self-test
_tests = [
    ("After reasoning, the answer is C", "C"),
    ("The correct answer is B.",         "B"),
    ("#### D",                           "D"),
    ("(A)",                              "A"),
    ("**B**",                            "B"),
]
ok = all(extract_pred_answer(t)==e for t,e in _tests)
print("Extractor:", "ALL PASSED" if ok else "FAILURES DETECTED")
for txt, exp in _tests:
    got = extract_pred_answer(txt)
    print(f"  {'OK' if got==exp else 'FAIL'}  '{txt}' -> '{got}'")

Extractor: ALL PASSED
  OK  'After reasoning, the answer is C' -> 'C'
  OK  'The correct answer is B.' -> 'B'
  OK  '#### D' -> 'D'
  OK  '(A)' -> 'A'
  OK  '**B**' -> 'B'


In [6]:
# CELL 7 -- Load 3B model (BASE only -- NO LoRA, NO fine-tuning)
# Critical difference from Notebook 10: no PeftModel, no adapter, pure base weights.
# We load the SAME base model that was used as the guide in Notebook 10,
# but here it acts as a standalone solver -- no planning, no split.
print(f"Loading: {CONFIG['model_name']}")
print("Adapter: NONE -- base model only, no task-specific training")

model_tok = AutoTokenizer.from_pretrained(CONFIG["model_name"], trust_remote_code=True)
if model_tok.pad_token is None:
    model_tok.pad_token = model_tok.eos_token

model_3b = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_name"],
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
).eval()

if torch.cuda.is_available():
    used  = torch.cuda.memory_allocated() / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"Model VRAM : {used:.2f} GB / {total:.1f} GB")
    print(f"Headroom   : {total - used:.1f} GB")

print(f"Compute per question: {CONFIG['model_params_B']}B x {CONFIG['n_votes']} = {CONFIG['model_params_B']*CONFIG['n_votes']}B param-passes")
print("Model ready -- no fine-tuning, no adapter")

Loading: Qwen/Qwen2.5-3B-Instruct
Adapter: NONE -- base model only, no task-specific training


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model VRAM : 5.75 GB / 15.9 GB
Headroom   : 10.1 GB
Compute per question: 3.0B x 5 = 15.0B param-passes
Model ready -- no fine-tuning, no adapter


In [7]:
# CELL 8 -- Generation functions
# ARC-Challenge science MCQ prompts for the base 3B solver.
# Same SOLVE_BASELINE_SYSTEM used in Notebook 10's baseline condition,
# applied here to the 3B model for a fair apples-to-apples comparison.

SOLVE_SYSTEM = (
    "You are a science question answering assistant."
    " Read the question carefully. Use your knowledge to pick the best answer."
    " No markdown. Plain text only."
    " Your absolute last line must be exactly: The answer is [letter]"
)

REFINER_SYSTEM = (
    "You are a careful science reasoning checker."
    " You are given a science question and a list of candidate answers that are tied."
    " Reason step by step about which answer is most scientifically accurate."
    " Your absolute last line must be exactly: The answer is [letter]"
)

def run_3b(messages, max_tokens, temperature):
    prompt = model_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = model_tok(prompt, return_tensors="pt", truncation=True, max_length=1024)
    dev    = next(model_3b.parameters()).device
    inputs = {k: v.to(dev) for k, v in inputs.items()}
    with torch.no_grad():
        out = model_3b.generate(
            **inputs,
            max_new_tokens     = max_tokens,
            temperature        = max(temperature, 0.05),
            do_sample          = True,
            top_p              = 0.92,
            top_k              = 40,
            pad_token_id       = model_tok.eos_token_id,
            repetition_penalty = 1.15,
        )
    new_toks = out[0][inputs["input_ids"].shape[1]:]
    return model_tok.decode(new_toks, skip_special_tokens=True).strip()

def generate_solve(question):
    return run_3b(
        [{"role":"system","content":SOLVE_SYSTEM},
         {"role":"user",  "content":question}],
        max_tokens  = CONFIG["max_new_tokens"],
        temperature = CONFIG["vote_temperature"],
    )

def generate_refine(question, candidates):
    cands   = ", ".join(sorted(set(c for c in candidates if c)))
    content = (f"{question}\n\nPrevious attempts gave different answers: {cands}\n"
               f"Re-reason carefully and pick the single best letter:")
    return run_3b(
        [{"role":"system","content":REFINER_SYSTEM},
         {"role":"user",  "content":content}],
        max_tokens  = CONFIG["max_new_tokens"],
        temperature = CONFIG["refiner_temperature"],
    )

print("Generation functions ready")
print(f"  generate_solve()   -- 3B base, temp {CONFIG['vote_temperature']}, no plan")
print(f"  generate_refine()  -- 3B base, temp {CONFIG['refiner_temperature']}, tie-breaker")

Generation functions ready
  generate_solve()   -- 3B base, temp 0.4, no plan
  generate_refine()  -- 3B base, temp 0.3, tie-breaker


In [8]:
# CELL 9 -- Voting logic (identical to Notebook 10)
def vote_and_decide(answers, question, gt_answer=None):
    valid = [a for a in answers if a and a.strip()]
    if not valid: valid = answers
    vote_counts = Counter(valid)
    most_common = vote_counts.most_common()
    top_answer  = most_common[0][0]
    top_count   = most_common[0][1]
    m total       = len(answers)

    correct_votes    = vote_counts.get(gt_answer, 0) if gt_answer else 0
    vote_consistency = correct_votes / total
    is_majority      = (len(most_common) == 1 or top_count > most_common[1][1])
    refiner_used = False; refiner_correct = None

    if is_majority:
        final = top_answer; strategy = "majority"
        conf  = round(top_count / total, 4); wasted = total - top_count
    else:
        ref_raw         = generate_refine(question, list(answers))
        ref_ans         = extract_pred_answer(ref_raw)
        refiner_used    = True
        refiner_correct = (ref_ans == gt_answer) if gt_answer else None
        all_votes   = answers + [ref_ans]
        new_counts  = Counter(all_votes)
        new_common  = new_counts.most_common()
        new_top     = new_common[0][0]
        new_top_c   = new_common[0][1]
        still_tied  = len(new_common) > 1 and new_top_c == new_common[1][1]
        final       = new_top
        strategy    = "coin_flip" if still_tied else "refiner_tiebreak"
        conf        = round(new_top_c / len(all_votes), 4)
        vote_counts = new_counts; total = len(all_votes)
        correct_votes    = new_counts.get(gt_answer, 0) if gt_answer else 0
        vote_consistency = correct_votes / total
        wasted           = total - new_top_c

    return {
        "final_answer"    : final, "strategy":strategy, "confidence":conf,
        "vote_counts"     : dict(vote_counts), "correct_votes":correct_votes,
        "total_votes"     : total, "vote_consistency":round(vote_consistency,4),
        "wasted_votes"    : wasted, "refiner_used":refiner_used,
        "refiner_correct" : refiner_correct,
    }

print("Voting logic ready (majority / refiner_tiebreak / coin_flip)")

Voting logic ready (majority / refiner_tiebreak / coin_flip)


In [9]:
# CELL 10 -- Single question test
print("=" * 65)
print("SINGLE QUESTION TEST  (ARC-Challenge -- 3B Ceiling)")
print("=" * 65)
item = test_data[0]
q    = item["question"]
gt   = extract_gt_answer(item["answer"])
print(f"Question :\n{q}")
print(f"GT Answer: {gt}")
print(f"\nRunning {CONFIG['n_votes']} votes (3B base, no plan, temp={CONFIG['vote_temperature']})...")

votes_raw = []
for i in range(CONFIG["n_votes"]):
    raw  = generate_solve(q)
    pred = extract_pred_answer(raw)
    votes_raw.append(pred)
    print(f"  Vote {i+1}: '{pred}'  |  raw[:80]: {raw[:80]}")

dec = vote_and_decide(votes_raw, q, gt)
print(f"\n  Result    : {dec['final_answer']}  (GT: {gt})  {'CORRECT' if dec['final_answer']==gt else 'WRONG'}")
print(f"  Strategy  : {dec['strategy']}")
print(f"  Confidence: {dec['confidence']}")
print(f"  Correct votes: {dec['correct_votes']}/{dec['total_votes']}")
print(f"  Vote counts  : {dec['vote_counts']}")
print("\nTest done -- run Cell 11 for full 900-question evaluation")

SINGLE QUESTION TEST  (ARC-Challenge -- 3B Ceiling)
Question :
When cold temperatures are produced in a chemical reaction, the reaction is known as

Options:
A) exothermic.
B) endothermic.
C) suspension.
D) vaporization.
GT Answer: B

Running 5 votes (3B base, no plan, temp=0.4)...
  Vote 1: 'B'  |  raw[:80]: B) endothermic.
  Vote 2: 'B'  |  raw[:80]: B) endothermic.
  Vote 3: 'B'  |  raw[:80]: B) endothermic.
  Vote 4: 'B'  |  raw[:80]: B) endothermic.
  Vote 5: 'B'  |  raw[:80]: B) endothermic.

  Result    : B  (GT: B)  CORRECT
  Strategy  : majority
  Confidence: 1.0
  Correct votes: 5/5
  Vote counts  : {'B': 5}

Test done -- run Cell 11 for full 900-question evaluation


In [10]:
# CELL 11 -- Full Evaluation Loop
# 900 questions, single ceiling_3b condition, mode stored per record.
# Checkpointing every save_every questions -- safe to interrupt and resume.

print(f"ARC-Challenge 3B Ceiling: {len(test_data)} questions")
print(f"Model  : {CONFIG['model_name']} (base, no LoRA)")
print(f"Votes  : {CONFIG['n_votes']} x temp {CONFIG['vote_temperature']}")
print(f"Compute: {CONFIG['model_params_B']}B x {CONFIG['n_votes']} = {CONFIG['model_params_B']*CONFIG['n_votes']}B param-passes")
print(f"Random chance: {CONFIG['random_chance']}% (4-option MCQ A-D)")
print("-" * 65)

results   = []
start_idx = 0

if os.path.exists(CONFIG["checkpoint_file"]):
    with open(CONFIG["checkpoint_file"]) as f:
        ckpt = json.load(f)
    start_idx = ckpt.get("last_index", 0)
    if os.path.exists(CONFIG["results_file"]):
        with open(CONFIG["results_file"]) as f:
            results = [json.loads(l) for l in f if l.strip()]
    print(f"Resumed from index {start_idx} ({len(results)} saved)")
else:
    print("Starting fresh")

t0 = time.time()

for idx in tqdm(range(start_idx, len(test_data)), desc="ARC 3B Ceiling"):
    item      = test_data[idx]
    question  = item["question"]
    gt_answer = extract_gt_answer(item["answer"])

    try:
        votes_raw = [extract_pred_answer(generate_solve(question))
                     for _ in range(CONFIG["n_votes"])]
        dec = vote_and_decide(votes_raw, question, gt_answer)
        results.append({
            "mode"            : "ceiling_3b",
            "idx"             : idx,
            "question"        : question,
            "gt_answer"       : gt_answer,
            "final_answer"    : dec["final_answer"],
            "correct"         : dec["final_answer"] == gt_answer,
            "strategy"        : dec["strategy"],
            "confidence"      : dec["confidence"],
            "correct_votes"   : dec["correct_votes"],
            "total_votes"     : dec["total_votes"],
            "vote_consistency": dec["vote_consistency"],
            "wasted_votes"    : dec["wasted_votes"],
            "refiner_used"    : dec["refiner_used"],
            "refiner_correct" : dec["refiner_correct"],
            "vote_counts"     : dec["vote_counts"],
        })
    except RuntimeError as e:
        results.append({
            "mode":"ceiling_3b","idx":idx,"question":question,
            "gt_answer":gt_answer,"final_answer":"","correct":False,"strategy":"error",
            "confidence":0.0,"correct_votes":0,"total_votes":CONFIG["n_votes"],
            "vote_consistency":0.0,"wasted_votes":CONFIG["n_votes"],
            "refiner_used":False,"refiner_correct":None,"vote_counts":{},"error":str(e),
        })

    if (idx + 1) % CONFIG["save_every"] == 0:
        with open(CONFIG["results_file"],"w") as f:
            for r in results: f.write(json.dumps(r)+"\n")
        with open(CONFIG["checkpoint_file"],"w") as f:
            json.dump({"last_index":idx+1},f)
        acc  = sum(r["correct"] for r in results)/len(results)*100
        mins = (time.time()-t0)/60
        print(f"  [{idx+1:3d}/{len(test_data)}]  3B Ceiling: {acc:.1f}%  ({mins:.1f} min)")

# Final save
with open(CONFIG["results_file"],"w") as f:
    for r in results: f.write(json.dumps(r)+"\n")
with open(CONFIG["checkpoint_file"],"w") as f:
    json.dump({"last_index":len(test_data)},f)

correct = sum(r["correct"] for r in results)
acc     = correct/len(results)*100
print(f"\nEvaluation complete.")
print(f"  3B Ceiling : {correct}/{len(results)} = {acc:.1f}%")
print(f"  Pipeline   : {CONFIG['known_pipeline_acc']}%  (known, 10.5B, N=900)")
print(f"  Baseline   : {CONFIG['known_baseline_acc']}%  (known, 7.5B, N=900)")

ARC-Challenge 3B Ceiling: 900 questions
Model  : Qwen/Qwen2.5-3B-Instruct (base, no LoRA)
Votes  : 5 x temp 0.4
Compute: 3.0B x 5 = 15.0B param-passes
Random chance: 25.0% (4-option MCQ A-D)
-----------------------------------------------------------------
Starting fresh


ARC 3B Ceiling:   0%|          | 0/900 [00:00<?, ?it/s]

  [ 25/900]  3B Ceiling: 68.0%  (1.7 min)
  [ 50/900]  3B Ceiling: 64.0%  (3.2 min)
  [ 75/900]  3B Ceiling: 65.3%  (5.4 min)
  [100/900]  3B Ceiling: 70.0%  (7.2 min)
  [125/900]  3B Ceiling: 70.4%  (8.4 min)
  [150/900]  3B Ceiling: 69.3%  (10.2 min)
  [175/900]  3B Ceiling: 68.6%  (11.3 min)
  [200/900]  3B Ceiling: 67.5%  (12.7 min)
  [225/900]  3B Ceiling: 67.6%  (13.7 min)
  [250/900]  3B Ceiling: 67.2%  (15.0 min)
  [275/900]  3B Ceiling: 65.8%  (16.5 min)
  [300/900]  3B Ceiling: 66.3%  (17.8 min)
  [325/900]  3B Ceiling: 65.5%  (19.2 min)
  [350/900]  3B Ceiling: 64.9%  (20.3 min)
  [375/900]  3B Ceiling: 64.5%  (21.4 min)
  [400/900]  3B Ceiling: 64.8%  (23.2 min)
  [425/900]  3B Ceiling: 64.2%  (24.7 min)
  [450/900]  3B Ceiling: 65.8%  (26.1 min)
  [475/900]  3B Ceiling: 65.3%  (27.5 min)
  [500/900]  3B Ceiling: 65.2%  (28.8 min)
  [525/900]  3B Ceiling: 65.3%  (30.4 min)
  [550/900]  3B Ceiling: 65.3%  (31.9 min)
  [575/900]  3B Ceiling: 65.9%  (33.1 min)
  [600/900]  3B 

In [11]:
# CELL 12 -- ANGLE 1: COMPUTE EFFICIENCY
ceiling_compute  = CONFIG["model_params_B"] * CONFIG["n_votes"]   # 15.0B
pipeline_compute = CONFIG["known_pipeline_B"]                      # 10.5B
baseline_compute = CONFIG["known_baseline_B"]                      #  7.5B

ceiling_acc  = sum(r["correct"] for r in results)/len(results)*100
pipeline_acc = CONFIG["known_pipeline_acc"]
baseline_acc = CONFIG["known_baseline_acc"]
random_chance = CONFIG["random_chance"]

ceiling_eff  = ceiling_acc  / ceiling_compute
pipeline_eff = pipeline_acc / pipeline_compute
baseline_eff = baseline_acc / baseline_compute

n = len(results); N = CONFIG["n_votes"]
ceiling_wasted = sum(r["wasted_votes"] for r in results)
strategy_stats = {}
for r in results:
    s = r["strategy"]
    if s not in strategy_stats: strategy_stats[s] = {"n":0,"correct":0}
    strategy_stats[s]["n"] += 1
    if r["correct"]: strategy_stats[s]["correct"] += 1

ref_triggered = sum(r["refiner_used"] for r in results)
ref_correct   = sum(1 for r in results if r["refiner_used"] and r.get("refiner_correct"))

print("=" * 68)
print("ANGLE 1 -- COMPUTE EFFICIENCY  (ARC-Challenge -- Three-Way Comparison)")
print("=" * 68)
print(f"  Random chance baseline: {random_chance}% (4 options A-D)")
print()
print(f"  {'Setup':<40} | {'Compute':>8} | {'Accuracy':>9} | {'Above Chance':>13} | {'Acc/B':>7}")
print(f"  {'-'*40}-+-{'-'*8}-+-{'-'*9}-+-{'-'*13}-+-{'-'*7}")
print(f"  {'Baseline  (1.5B x 5)':<40} | {baseline_compute:>6.1f}B  | {baseline_acc:>8.1f}% | {baseline_acc-random_chance:>+12.1f}% | {baseline_eff:>6.3f}")
print(f"  {'Pipeline  (3B+LoRA x1 + 1.5B x5)':<40} | {pipeline_compute:>6.1f}B  | {pipeline_acc:>8.1f}% | {pipeline_acc-random_chance:>+12.1f}% | {pipeline_eff:>6.3f}")
print(f"  {'Ceiling   (3B base x 5)  <- THIS':<40} | {ceiling_compute:>6.1f}B  | {ceiling_acc:>8.1f}% | {ceiling_acc-random_chance:>+12.1f}% | {ceiling_eff:>6.3f}")
print()
print(f"  Pipeline vs Ceiling  : {pipeline_acc - ceiling_acc:+.1f} pts  (pipeline uses {ceiling_compute - pipeline_compute:.1f}B LESS)")
print(f"  Ceiling  vs Baseline : {ceiling_acc  - baseline_acc:+.1f} pts  (+{ceiling_compute - baseline_compute:.1f}B more)")
print(f"  Pipeline vs Baseline : {pipeline_acc - baseline_acc:+.1f} pts  (+{pipeline_compute - baseline_compute:.1f}B more)")
print()
if pipeline_acc >= ceiling_acc:
    print(f"  RESULT: Pipeline MATCHES or BEATS ceiling at 30% lower cost.")
    print(f"  Fine-tuned verbal planning outperforms raw model capacity on ARC-Challenge.")
else:
    gap  = ceiling_acc - pipeline_acc
    frac = (pipeline_acc - baseline_acc) / max(ceiling_acc - baseline_acc, 0.01) * 100
    print(f"  RESULT: 3B ceiling leads pipeline by {gap:.1f} pts at 43% higher cost.")
    print(f"  Pipeline recovers {frac:.0f}% of ceiling gain at {pipeline_compute/ceiling_compute*100:.0f}% of ceiling cost.")
    print(f"  Note: guide was fine-tuned on math (GSM8K), not science.")
    print(f"  A domain-matched fine-tuned guide may close this gap further.")

print(f"\n  Wasted votes -- Ceiling: {ceiling_wasted}/{n*N} ({ceiling_wasted/(n*N)*100:.1f}%)")
if ref_triggered > 0:
    print(f"  Refiner: triggered {ref_triggered}, correct {ref_correct}")
print(f"\n  Strategy breakdown:")
for s,v in sorted(strategy_stats.items(), key=lambda x:-x[1]['n']):
    acc_s = v['correct']/v['n']*100 if v['n'] else 0
    print(f"    {s:<22}  n={v['n']:4d}  acc={acc_s:.1f}%")

angle1 = {
    "dataset":"ARC-Challenge","experiment":"ceiling_3b","n_questions":n,
    "random_chance":random_chance,
    "ceiling_accuracy":round(ceiling_acc,2),"pipeline_accuracy":pipeline_acc,"baseline_accuracy":baseline_acc,
    "ceiling_compute_B":ceiling_compute,"pipeline_compute_B":pipeline_compute,"baseline_compute_B":baseline_compute,
    "ceiling_efficiency":round(ceiling_eff,4),"pipeline_efficiency":round(pipeline_eff,4),"baseline_efficiency":round(baseline_eff,4),
    "ceiling_wasted_votes":ceiling_wasted,"refiner_triggered":ref_triggered,"refiner_correct":ref_correct,
    "strategy_breakdown":strategy_stats,"pipeline_beats_ceiling":pipeline_acc >= ceiling_acc,
}
with open(CONFIG["angle1_file"],"w") as f: json.dump(angle1,f,indent=2)
print(f"\nSaved -> {CONFIG['angle1_file']}")

ANGLE 1 -- COMPUTE EFFICIENCY  (ARC-Challenge -- Three-Way Comparison)
  Random chance baseline: 25.0% (4 options A-D)

  Setup                                    |  Compute |  Accuracy |  Above Chance |   Acc/B
  -----------------------------------------+----------+-----------+---------------+--------
  Baseline  (1.5B x 5)                     |    7.5B  |     71.7% |        +46.7% |  9.560
  Pipeline  (3B+LoRA x1 + 1.5B x5)         |   10.5B  |     80.0% |        +55.0% |  7.619
  Ceiling   (3B base x 5)  <- THIS         |   15.0B  |     66.8% |        +41.8% |  4.452

  Pipeline vs Ceiling  : +13.2 pts  (pipeline uses 4.5B LESS)
  Ceiling  vs Baseline : -4.9 pts  (+7.5B more)
  Pipeline vs Baseline : +8.3 pts  (+3.0B more)

  RESULT: Pipeline MATCHES or BEATS ceiling at 30% lower cost.
  Fine-tuned verbal planning outperforms raw model capacity on ARC-Challenge.

  Wasted votes -- Ceiling: 263/4500 (5.8%)
  Refiner: triggered 1, correct 0

  Strategy breakdown:
    majority         

In [12]:
# CELL 13 -- ANGLE 2: VOTE CONSISTENCY + POSITION BIAS (A-D)
cons_scores = [r["vote_consistency"] for r in results]
mean_cons   = np.mean(cons_scores)

def bucket(scores):
    return {
        "all_wrong  (0%)"  : sum(1 for s in scores if s == 0.0),
        "low       (1-39%)": sum(1 for s in scores if 0.0 < s < 0.4),
        "medium  (40-79%)" : sum(1 for s in scores if 0.4 <= s < 0.8),
        "high   (80-100%)" : sum(1 for s in scores if s >= 0.8),
    }

dist      = bucket(cons_scores)
corr_cons = [r["vote_consistency"] for r in results if r["correct"]]

# Position bias: which letter does the 3B base model prefer?
letter_votes      = defaultdict(int)
total_final_votes = 0
for r in results:
    for letter, count in r["vote_counts"].items():
        if letter in VALID_LETTERS:
            letter_votes[letter] += count
            total_final_votes    += count

# Known pipeline values from Notebook 10 N=900 run
pipeline_cons = 0.800   # pipeline accuracy = 80.0%, approximate consistency
baseline_cons = 0.717   # baseline accuracy = 71.7%, approximate consistency
pipeline_dist = {"all_wrong  (0%)":None,"low       (1-39%)":None,"medium  (40-79%)":None,"high   (80-100%)":None}
# Known pipeline position bias from Notebook 10: none detected on ARC-Challenge
# Known baseline position bias from Notebook 10: none detected

print("=" * 65)
print("ANGLE 2 -- VOTE CONSISTENCY  (ARC-Challenge -- Three-Way)")
print("=" * 65)
print(f"  Mean correct-vote ratio (out of {CONFIG['n_votes']} per question):")
print(f"    Baseline (1.5B x5) : {baseline_cons*100:.1f}%  ({baseline_cons*5:.2f}/5 avg)  [Notebook 10]")
print(f"    Pipeline (3B+LoRA) : {pipeline_cons*100:.1f}%  ({pipeline_cons*5:.2f}/5 avg)  [Notebook 10]")
print(f"    Ceiling  (3B base) : {mean_cons*100:.1f}%  ({mean_cons*5:.2f}/5 avg)  <- THIS")

print(f"\n  Distribution (3B Ceiling):")
print(f"  {'Bucket':<22} | {'Ceiling':>8} | {'Count':>8}")
print(f"  {'-'*22}-+-{'-'*8}-+-{'-'*8}")
for bkt in ["all_wrong  (0%)","low       (1-39%)","medium  (40-79%)","high   (80-100%)"]:
    cv = dist[bkt]
    print(f"  {bkt:<22} | {cv:>8} | {cv:>8}")

print(f"\n  Position Bias -- Option Letter Distribution:")
print(f"  Expected: 25.0% per option (uniform for 4-choice MCQ)")
print(f"  {'Letter':<6} | {'Ceiling':>8} | {'Expected':>8}")
print(f"  {'-'*6}-+-{'-'*8}-+-{'-'*8}")
for letter in sorted(VALID_LETTERS):
    ceil_pct = letter_votes.get(letter,0)/max(total_final_votes,1)*100
    flag = "  <- bias" if ceil_pct > 32 else ("  <- low" if ceil_pct < 18 else "")
    print(f"  {letter:<6} | {ceil_pct:>7.1f}% | {'25.0%':>8}{flag}")

max_letter = max(letter_votes, key=lambda l: letter_votes.get(l,0)) if letter_votes else "?"
max_pct    = letter_votes.get(max_letter,0)/max(total_final_votes,1)*100
if max_pct > 32:
    print(f"\n  ⚠ Position bias detected: ceiling model favours option {max_letter} ({max_pct:.1f}% vs 25% expected)")
else:
    print(f"\n  No strong position bias detected (max option: {max_letter} at {max_pct:.1f}%)")

if corr_cons:
    print(f"\n  Correct questions: ceiling consistency = {np.mean(corr_cons)*100:.1f}% (n={len(corr_cons)})")

# All-wrong count
all_wrong = dist["all_wrong  (0%)"]
high_agree = dist["high   (80-100%)"]
print(f"\n  All-wrong questions (0/5 votes correct)  : {all_wrong}")
print(f"  High-agreement questions (4-5/5 correct): {high_agree}")

angle2 = {
    "dataset":"ARC-Challenge","experiment":"ceiling_3b","n_questions":len(results),
    "ceiling_mean_consistency":round(mean_cons,4),
    "pipeline_mean_consistency":pipeline_cons,"baseline_mean_consistency":baseline_cons,
    "ceiling_distribution":dist,
    "ceiling_letter_dist":{l:round(letter_votes.get(l,0)/max(total_final_votes,1)*100,2) for l in sorted(VALID_LETTERS)},
    "ceiling_correct_q_consistency":round(np.mean(corr_cons),4) if corr_cons else 0,
    "all_wrong":all_wrong,"high_agreement":high_agree,
}
with open(CONFIG["angle2_file"],"w") as f: json.dump(angle2,f,indent=2)
print(f"\nSaved -> {CONFIG['angle2_file']}")

ANGLE 2 -- VOTE CONSISTENCY  (ARC-Challenge -- Three-Way)
  Mean correct-vote ratio (out of 5 per question):
    Baseline (1.5B x5) : 71.7%  (3.58/5 avg)  [Notebook 10]
    Pipeline (3B+LoRA) : 80.0%  (4.00/5 avg)  [Notebook 10]
    Ceiling  (3B base) : 63.5%  (3.18/5 avg)  <- THIS

  Distribution (3B Ceiling):
  Bucket                 |  Ceiling |    Count
  -----------------------+----------+---------
  all_wrong  (0%)        |      283 |      283
  low       (1-39%)      |       26 |       26
  medium  (40-79%)       |       42 |       42
  high   (80-100%)       |      549 |      549

  Position Bias -- Option Letter Distribution:
  Expected: 25.0% per option (uniform for 4-choice MCQ)
  Letter |  Ceiling | Expected
  -------+----------+---------
  A      |    24.1% |    25.0%
  B      |    24.7% |    25.0%
  C      |    28.2% |    25.0%
  D      |    23.0% |    25.0%
  E      |     0.0% |    25.0%  <- low

  No strong position bias detected (max option: C at 28.2%)

  Correct ques

In [13]:
# CELL 14 -- ANGLE 3: CONFIDENCE CALIBRATION
def calibration_report(res, label):
    buckets = [
        ("Very High  (>=0.80)", lambda c: c >= 0.80, 0.90),
        ("High       (0.60-0.80)", lambda c: 0.60 <= c < 0.80, 0.70),
        ("Medium     (0.40-0.60)", lambda c: 0.40 <= c < 0.60, 0.50),
        ("Low        (<0.40)",  lambda c: c < 0.40,  0.25),
    ]
    n_total = len(res); ece = 0.0; calib_out = []
    false_conf = sum(1 for r in res if r["confidence"] >= 0.80 and not r["correct"])
    print(f"\n  [{label}]")
    print(f"  {'Confidence':<26} | {'N':>5} | {'Accuracy':>9} | {'Expected':>9} | {'Gap':>6} | Cal?")
    print(f"  {'-'*26}-+-{'-'*5}-+-{'-'*9}-+-{'-'*9}-+-{'-'*6}-+----")
    for name, cond, mid in buckets:
        subset = [r for r in res if cond(r["confidence"])]
        if not subset:
            print(f"  {name:<26} | {'--':>5} | {'--':>9} | {mid*100:>8.0f}% | {'--':>6} |"); continue
        n=len(subset); acc=sum(r["correct"] for r in subset)/n; gap=abs(acc-mid)
        ece += (n/n_total)*gap; flag="Good" if gap<0.15 else "Poor"
        print(f"  {name:<26} | {n:>5} | {acc*100:>8.1f}% | {mid*100:>8.0f}% | {gap:>6.3f} | {flag}")
        calib_out.append({"bucket":name,"count":n,"accuracy":round(acc,4),"expected":mid,"gap":round(gap,4)})
    hc = [r for r in res if r["confidence"] >= 0.80]
    hc_acc = sum(r["correct"] for r in hc)/max(1,len(hc))*100
    print(f"  {'ECE (lower=better)':<26}   {ece:.4f}")
    print(f"  High-conf questions : {len(hc)}  |  Accuracy when confident: {hc_acc:.1f}%")
    print(f"  Confidently WRONG   : {false_conf}")
    return ece, calib_out, false_conf

# Known ECE from Notebook 10 N=900 pipeline run
known_pipe_ece = 0.0990   # pipeline ECE on ARC-Challenge
known_base_ece = 0.1800   # baseline ECE on ARC-Challenge (approx)

print("=" * 65)
print("ANGLE 3 -- CONFIDENCE CALIBRATION  (ARC-Challenge -- 3B Ceiling)")
print("=" * 65)
ceiling_ece, ceiling_calib, ceiling_false = calibration_report(results, "3B CEILING (this run)")
print(f"\n  ECE -- Three-Way:")
print(f"    Baseline (1.5B x5) : {known_base_ece:.4f}  [Notebook 10]")
print(f"    Pipeline (3B+LoRA) : {known_pipe_ece:.4f}  [Notebook 10]")
print(f"    Ceiling  (3B base) : {ceiling_ece:.4f}  <- THIS")
print(f"\n  Ceiling vs Pipeline: {ceiling_ece - known_pipe_ece:+.4f} ({'worse' if ceiling_ece > known_pipe_ece else 'better'})")
print(f"  False confidence (ceiling): {ceiling_false}")

angle3 = {
    "dataset":"ARC-Challenge","experiment":"ceiling_3b","n_questions":len(results),
    "ceiling_ece":round(ceiling_ece,4),"pipeline_ece":known_pipe_ece,"baseline_ece":known_base_ece,
    "ceiling_false_confidence":ceiling_false,"ceiling_calibration":ceiling_calib,
}
with open(CONFIG["angle3_file"],"w") as f: json.dump(angle3,f,indent=2)
print(f"\nSaved -> {CONFIG['angle3_file']}")

ANGLE 3 -- CONFIDENCE CALIBRATION  (ARC-Challenge -- 3B Ceiling)

  [3B CEILING (this run)]
  Confidence                 |     N |  Accuracy |  Expected |    Gap | Cal?
  ---------------------------+-------+-----------+-----------+--------+----
  Very High  (>=0.80)        |   821 |     66.9% |       90% |  0.231 | Poor
  High       (0.60-0.80)     |    40 |     65.0% |       70% |  0.050 | Good
  Medium     (0.40-0.60)     |    19 |     63.2% |       50% |  0.132 | Good
  Low        (<0.40)         |    20 |     70.0% |       25% |  0.450 | Poor
  ECE (lower=better)           0.2260
  High-conf questions : 821  |  Accuracy when confident: 66.9%
  Confidently WRONG   : 272

  ECE -- Three-Way:
    Baseline (1.5B x5) : 0.1800  [Notebook 10]
    Pipeline (3B+LoRA) : 0.0990  [Notebook 10]
    Ceiling  (3B base) : 0.2260  <- THIS

  Ceiling vs Pipeline: +0.1270 (worse)
  False confidence (ceiling): 272

Saved -> /kaggle/working/arc_ceiling/angle3_confidence_calibration.json


In [14]:
# CELL 15 -- Full Three-Way Summary Table
with open(CONFIG["angle1_file"]) as f: a1 = json.load(f)
with open(CONFIG["angle2_file"]) as f: a2 = json.load(f)
with open(CONFIG["angle3_file"]) as f: a3 = json.load(f)

ca = a1["ceiling_accuracy"]
pa = a1["pipeline_accuracy"]
ba = a1["baseline_accuracy"]

print("=" * 75)
print("  ARC-Challenge -- THREE-WAY: BASELINE / PIPELINE / 3B CEILING")
print(f"  N={a1['n_questions']} questions  |  Seed={CONFIG['random_seed']}  |  Qwen2.5 model family")
print(f"  Random chance: {CONFIG['random_chance']}% (4-option MCQ A-D)")
print("=" * 75)

rows = [
    ["Metric",                  "Baseline",          "Our Pipeline",         "3B Ceiling (this)"],
    ["Model",                   "Qwen2.5-1.5B x5",  "3B LoRA + 1.5B x5",   "Qwen2.5-3B base x5"],
    ["Fine-Tuning",             "None",              "LoRA on GSM8K",        "None"],
    ["Compute (param-passes)",  "7.5B",              "10.5B",                "15.0B"],
    ["─────────────────────",   "───────────────",   "───────────────────",  "──────────────────"],
    ["Overall Accuracy",        f"{ba:.1f}%",        f"{pa:.1f}%",           f"{ca:.1f}%"],
    ["Above Random Chance",     f"+{ba-25:.1f} pts", f"+{pa-25:.1f} pts",    f"+{ca-25:.1f} pts"],
    ["vs Baseline",             "—",                 f"+{pa-ba:.1f} pts",    f"+{ca-ba:.1f} pts"],
    ["Pipeline vs Ceiling",     "—",                 f"{'WINS' if pa>=ca else 'loses'} {abs(pa-ca):.1f} pts","←"],
    ["Acc / Billion passes",    f"{ba/7.5:.3f}",     f"{pa/10.5:.3f}",       f"{ca/15.0:.3f}"],
    ["─────────────────────",   "───────────────",   "───────────────────",  "──────────────────"],
    ["Vote Consistency",        f"{a2['baseline_mean_consistency']*100:.1f}%",
                                 f"{a2['pipeline_mean_consistency']*100:.1f}%",
                                 f"{a2['ceiling_mean_consistency']*100:.1f}%"],
    ["ECE (lower=better)",      f"{a3['baseline_ece']:.4f}",
                                 f"{a3['pipeline_ece']:.4f}",
                                 f"{a3['ceiling_ece']:.4f}"],
    ["False Confidence",        "—",                 "—",                    str(a3["ceiling_false_confidence"])],
]

col_w = [26, 20, 22, 20]
sep   = "-+-".join("-"*w for w in col_w)
for i, row in enumerate(rows):
    if "─────" in row[0]:
        print("  " + sep); continue
    line = " | ".join(str(c).ljust(col_w[j]) for j,c in enumerate(row))
    print("  " + line)
    if i == 0: print("  " + sep)

print()
print("  Position Bias (3B Ceiling, expected 25.0% per option):")
for letter, pct in sorted(a2["ceiling_letter_dist"].items()):
    bar  = "█" * int(pct/2)
    flag = " <- bias" if pct > 32 else (" <- low" if pct < 18 else "")
    print(f"    {letter}: {pct:5.1f}%  {bar}{flag}")

print()
print("=" * 75)
if pa >= ca:
    print(f"  VERDICT: Pipeline BEATS the 3B ceiling (+{pa-ca:.1f} pts) at 30% lower cost.")
    print(f"  Fine-tuned verbal planning outperforms raw model capacity on ARC-Challenge.")
    print(f"  Model size alone does NOT explain the guided pipeline's gains.")
else:
    gap  = ca - pa
    frac = (pa - ba) / max(ca - ba, 0.01) * 100
    print(f"  VERDICT: 3B ceiling leads pipeline by {gap:.1f} pts at 43% higher cost.")
    print(f"  Pipeline delivers {frac:.0f}% of ceiling gain at {pa/ca*100:.0f}% of ceiling cost.")
    print(f"  The 3B base model has broader science knowledge that raw scale can unlock.")
    print(f"  Pipeline's math-only fine-tuning partially but not fully closes the domain gap.")
print("=" * 75)

full = {
    "dataset":"ARC-Challenge","seed":CONFIG["random_seed"],"n_questions":a1["n_questions"],
    "conditions":{
        "baseline":{"compute_B":7.5, "accuracy":ba,"model":"Qwen2.5-1.5B x5","fine_tuned":False},
        "pipeline":{"compute_B":10.5,"accuracy":pa,"model":"3B LoRA + 1.5B x5","fine_tuned":True},
        "ceiling" :{"compute_B":15.0,"accuracy":ca,"model":"3B base x5","fine_tuned":False},
    },
    "pipeline_beats_ceiling":pa >= ca,
    "angle1":a1,"angle2":a2,"angle3":a3,
}
with open(CONFIG["report_file"],"w") as f: json.dump(full,f,indent=2)
print(f"\nAll results saved to {OUTPUT_DIR}/")
print("Files: results.jsonl · eval_report.json · angle1/2/3.json")

  ARC-Challenge -- THREE-WAY: BASELINE / PIPELINE / 3B CEILING
  N=900 questions  |  Seed=42  |  Qwen2.5 model family
  Random chance: 25.0% (4-option MCQ A-D)
  Metric                     | Baseline             | Our Pipeline           | 3B Ceiling (this)   
  ---------------------------+----------------------+------------------------+---------------------
  Model                      | Qwen2.5-1.5B x5      | 3B LoRA + 1.5B x5      | Qwen2.5-3B base x5  
  Fine-Tuning                | None                 | LoRA on GSM8K          | None                
  Compute (param-passes)     | 7.5B                 | 10.5B                  | 15.0B               
  ---------------------------+----------------------+------------------------+---------------------
  Overall Accuracy           | 71.7%                | 80.0%                  | 66.8%               
  Above Random Chance        | +46.7 pts            | +55.0 pts              | +41.8 pts           
  vs Baseline                | —        